In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import pyodbc
import plotly.io as pio
import numpy as np
import statsmodels.api as sm
import us

**LOAD SQL FILE**

In [ ]:
conn = pyodbc.connect("DRIVER={ODBC Driver 17 for SQL Server};"
                      "SERVER=localhost;"
                      "DATABASE=MotoDB;"
                      "Trusted_Connection=yes;")

In [ ]:
df = pd.read_sql("SELECT * FROM MotoDB.dbo.VehicleSales", conn)

**DATA DESCRIBE**

In [ ]:
df.info()

In [ ]:
df.describe()

***DATA CLEANING & DATA WRANGLING***

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df.replace(['nan', 'NaN', '—'], np.nan, inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df["State"].unique()

In [ ]:
df["State"] = df["State"].apply(lambda x:us.states.lookup(str(x)))

In [ ]:
df['State'] = df['State'].astype(str)

In [ ]:
df["Transmission"] = df["Transmission"].apply(lambda x:str(x).lower())
df["Transmission"].value_counts()

In [ ]:
df = df[df['Transmission'] != 'sedan']

In [ ]:
df.drop(index=df[(df["Body"] == "navitgation")].index, inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
def ConditionValuePreparation(x):
    if x in [1, 2, 3, 4, 5]:
        x = x
    elif x == np.nan:
        x = np.nan
    else:
        x /= 10


In [ ]:
def ConditionValuePreparation(x):
    try:
        if x in [1, 2, 3, 4, 5]:
            x = x
        else:
            x /= 10
    except:
        x = np.nan
    
    return x


In [ ]:
df["ConditionValue"] = df["ConditionValue"].apply(lambda x:ConditionValuePreparation(x))
df["ConditionValue"]

In [ ]:
df.dropna(subset=['SaleDate'], inplace=True)

In [ ]:
df.dropna(subset=['Odometer'],inplace=True)

In [ ]:
df['ConditionValue'] = df['ConditionValue'].fillna(df.groupby('Year')['ConditionValue'].transform('mean'))

In [ ]:
df.dropna(subset=["Make","Model","Trim","Body"],how='all',inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
df.dropna(subset=["Model","Trim"],inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
df['Body'] = df['Body'].fillna(df.groupby('Make')['Body'].transform(lambda x: x.mode()[0] if not x.mode().empty else "sedan"))

In [ ]:
df["Transmission"] = df["Transmission"].fillna(df.groupby(["Make","Model"])["Transmission"].transform(lambda x: x.mode()[0] if not x.mode().empty else "automatic"))

In [ ]:
temp_state = df["State"].fillna(df.groupby(["Make","Model","Year"])["State"].transform(lambda x:x.mode()[0] if not x.mode().empty else "Florida"))
print(temp_state.value_counts())

In [ ]:
df["State"] = df["State"].fillna(df.groupby(["Make","Model","Year"])["State"].transform(lambda x:x.mode()[0] if not x.mode().empty else "Florida"))

In [ ]:
df["Color"] = df["Color"].fillna(df.groupby(['Make','Model'])['Color'].transform(lambda x:x.mode()[0] if not x.mode().empty else "white"))

In [ ]:
df["Interior"] = df["Interior"].fillna(df.groupby(['Color','Body'])['Interior'].transform(lambda x:x.mode()[0] if not x.mode().empty else "black"))

In [ ]:
df.isnull().sum()

*****EDA*****

In [ ]:
top_10_years = df["Year"].value_counts().head(10).reset_index()
top_10_years


In [ ]:
fig = px.bar(top_10_years, x="Year", y="count", text_auto=True, title="Top 10 Years Models")
fig.update_layout(xaxis=dict(tickmode="linear"))
fig.show()

In [ ]:
top_brands = df["Make"].value_counts().reset_index()
top_brands

In [ ]:
fig = px.bar(top_brands, x="Make", y="count", text_auto=True, title="Brands")
# fig.update_layout()
fig.show()

In [ ]:
ford_models = df[df["Make"] == "Ford"]["Model"].value_counts().reset_index()
fig = px.bar(ford_models, x="Model", y="count", text_auto=True, title="Ford Models")
# fig.update_layout()
fig.show()

In [ ]:
top_10_states = df["State"].value_counts().reset_index()
top_10_states["State"] = top_10_states["State"].apply(lambda x: str(x))
fig = px.bar(top_10_states, x="State", y="count", text_auto=True, title="Top 10 States")
fig.show()

In [ ]:
State_prices = df.groupby('State')['SellingPrice'].mean().reset_index().sort_values('SellingPrice')
fig = px.box(df.sample(1000), x="State", y="SellingPrice", color="State",title ="Price Distribution by State")
fig.update_xaxes(categoryorder='median ascending')
fig.show()

In [ ]:
state_avg = df.groupby('State')['SellingPrice'].mean().reset_index()
fig = px.bar(state_avg,x='State',y='SellingPrice',color='SellingPrice',title='Average Selling Price by State')
fig.update_layout(xaxis={'categoryorder':'total ascending'})
fig.show()

In [ ]:
fig = px.scatter(df.sample(100000),x = "SellingPrice",y = "MMR", title= "the different between them")
fig.show()

In [ ]:
df['price_different'] = df['SellingPrice'] - df['MMR']
fig = px.scatter(df,x='MMR',y='SellingPrice',trendline='ols', color='price_different',title="Selling Price vs MMR ")
fig.show()

In [ ]:
fig=px.scatter(df,x='Odometer',y=['SellingPrice','MMR'],opacity=0.5,title='Odometer vs (Selling Price & MMR)')
fig.show()

In [ ]:
top_condition = df.groupby(['Make', 'Model'])['ConditionValue'].mean().reset_index().sort_values(by='ConditionValue', ascending=False).head(30)
fig = px.treemap(top_condition, path=['Make', 'Model'], values='ConditionValue',
color='ConditionValue', color_continuous_scale='RdYlGn',
title="Average Condition by Make and Model")
fig.show()

In [ ]:
#df.to_csv('Cars_Processed_Data.csv', index=False, encoding='utf-8-sig')